# Chapter 12: Polytopes; Compact Convex Sets

Source orientation: printed Volume II pages 1-85; PDF pages 10-94. This notebook is an original visualization-first lesson based on the chapter structure and concepts, not a substitute scan or excerpt.

The chapter question is: how can we turn convex hulls, volume and area, regular polytopes, Euler characteristic, Cauchy rigidity, approximation, and isoperimetry into objects that can be drawn, measured, transformed, and checked? The answer throughout this notebook is to treat definitions as computational contracts. A convex body becomes hull data and supporting inequalities; a form becomes a symmetric matrix with a visible signature; a conic, sphere, or hyperbolic model becomes an object whose invariants can be probed by code.

The notebook is meant to stand on its own. It introduces the working vocabulary, builds diagrams from scratch, runs small numerical checks, and ends with a lab that can be modified without reopening the book. The source pages are used for orientation: they tell us which ideas belong together, where the chapter puts emphasis, and which examples are worth turning into inspectable experiments.


## Translation guide

- Objects: finite convex hulls, support lines, regular polygons and polyhedra, compact bodies, surface area, and approximating families.
- Invariants: convex hull containment, signed area, monotone improvement of polygonal approximations, and Euler characteristic checks.
- Main misconception to disarm: The common trap is to treat a polytope as a picture of flat faces rather than as a finite system of inequalities, vertices, incidence data, and limiting comparisons with smooth bodies.
- Computational rule of thumb: start from the algebraic representation, draw the geometric locus, then assert the quantity that should not change.

This translation guide is deliberately practical. It does not try to reproduce every proof. Instead it asks which parts of a proof have a state that can be inspected: an incidence relation, a sign pattern, a limiting family, a deformation, a distance comparison, or a rank calculation. When the theorem is global, the notebook uses examples as probes and labels them as probes. When the claim is an identity, the notebook makes the identity executable.


## Route through the chapter

1. Build a small dictionary of the chapter's objects and the numerical representation used in the notebook.
2. Draw the primary geometric situation with labels and stable axes.
3. Vary a parameter or compare related models so the invariant has something to resist.
4. Save artifacts under `artifacts/chapter-12` and display them inline.
5. Run sanity checks that assert the relevant residuals, distances, signatures, or incidence relations.

The point of this route is not speed. It is to make the reader's eye and the computer agree about what the geometry says. If a diagram is attractive but no invariant is named near it, it is not yet doing mathematical work. If a formula is true but nothing in the notebook lets the reader inspect the objects it relates, the lesson is too thin.


In [ ]:
from pathlib import Path
import sys

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the Geometry II book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER = 12
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / f"chapter-{CHAPTER:02d}"
FIGURE_ROOT = ARTIFACT_ROOT / "figures"
PLOT_ROOT = ARTIFACT_ROOT / "plots"
TABLE_ROOT = ARTIFACT_ROOT / "tables"
CHECK_ROOT = ARTIFACT_ROOT / "checks"
for root in [FIGURE_ROOT, PLOT_ROOT, TABLE_ROOT, CHECK_ROOT]:
    root.mkdir(parents=True, exist_ok=True)

print(f"Geometry II root: {BOOK_ROOT}")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from utils.artifacts import assert_artifacts, display_artifact, save_csv, save_json
from utils.geometry import (
    circle_orthogonality,
    conic_matrix,
    conic_residual,
    convex_hull_2d,
    cross_ratio,
    disk_rotation,
    euler_characteristic,
    ellipse_points,
    hyperbola_points,
    polar_line,
    poincare_distance,
    polygon_area,
    quadratic_signature,
    regular_polygon,
    sphere_grid,
    spherical_distance,
    stereographic_project,
)
from utils.plotting import COLORS, finish_axes, new_3d_axes, new_axes, plot_line, plot_points, plot_polyline, plot_unit_circle, save_figure, set_equal_3d

generated_artifacts = []


## Visualization storyboard

- Convex hull and regular polygon approximation plotted as inspectable planar objects.
- A wireframe cube/octahedron incidence check for Euler characteristic.
- A JSON/CSV audit of area convergence and V - E + F.

Each visual is paired with a check. The checks are intentionally small and readable: area ratios, matrix signatures, collinearity residuals, distance invariance, and orthogonality errors. This keeps the chapter honest. The plotted object is not a decoration; it is the front end for a reproducible computation.


In [ ]:
rng = np.random.default_rng(12)
cloud = rng.normal(size=(28, 2))
cloud[:, 0] *= 1.2
cloud[:, 1] *= 0.75
hull = convex_hull_2d(cloud)

fig, ax = new_axes(title="Convex hull as a finite certificate")
plot_points(ax, cloud, color=COLORS["blue"], size=32)
plot_polyline(ax, hull, closed=True, color=COLORS["orange"], linewidth=2.5, label="convex hull")
ax.legend(loc="upper right")
ax.set_xlabel("x")
ax.set_ylabel("y")
finish_axes(ax)
hull_path = FIGURE_ROOT / "convex-hull-finite-certificate.png"
save_figure(fig, hull_path)
generated_artifacts.append(hull_path)
display_artifact(hull_path)


In [ ]:
rows = []
fig, ax = new_axes(title="Regular polygons approximate the disk")
t = np.linspace(0, 2 * np.pi, 500)
ax.plot(np.cos(t), np.sin(t), color=COLORS["ink"], linewidth=2.0, label="unit circle")
for n, color in [(6, COLORS["red"]), (12, COLORS["orange"]), (24, COLORS["teal"]), (48, COLORS["purple"])]:
    poly = regular_polygon(n, phase=np.pi / n)
    area = abs(polygon_area(poly))
    rows.append({"n": n, "area": area, "area_error": math.pi - area, "relative_error": (math.pi - area) / math.pi})
    plot_polyline(ax, poly, closed=True, color=color, linewidth=1.5, label=f"{n}-gon")
ax.legend(loc="lower right", fontsize=8)
finish_axes(ax, margin=0.15)
approx_path = FIGURE_ROOT / "regular-polygons-approach-disk.png"
save_figure(fig, approx_path)
table_path = TABLE_ROOT / "regular-polygon-area-convergence.csv"
save_csv(rows, table_path)
generated_artifacts.extend([approx_path, table_path])
display_artifact(approx_path)


In [ ]:
euler_checks = {
    "cube": {"V": 8, "E": 12, "F": 6, "chi": euler_characteristic(8, 12, 6)},
    "octahedron": {"V": 6, "E": 12, "F": 8, "chi": euler_characteristic(6, 12, 8)},
    "tetrahedron": {"V": 4, "E": 6, "F": 4, "chi": euler_characteristic(4, 6, 4)},
    "area_convergence_last_error": rows[-1]["area_error"],
}
check_path = CHECK_ROOT / "polytope-area-euler-checks.json"
save_json(euler_checks, check_path)
generated_artifacts.append(check_path)
display_artifact(check_path)


## Concept frame

The central objects of this chapter can be read at three levels. First there is a synthetic level: points, lines, planes, spheres, tangencies, and intersections. Second there is an algebraic level: coordinates, matrices, determinants, ranks, signatures, and residuals. Third there is a metric or topological level when the chapter asks for length, area, angle, orientation, compactness, or connectedness. A standalone notebook has to keep those levels visible at the same time.

The diagrams below are therefore built from data rather than imported as fixed pictures. That choice matters. If the reader changes a parameter, the artifact changes with it and the check either continues to pass or reveals exactly which assumption was broken. This is especially useful in Berger's style of geometry, where an object is often introduced synthetically and then compared with an affine, projective, Euclidean, spherical, or hyperbolic model.

The chapter also rewards paying attention to degeneracy. Degenerate cases are not annoyances pushed to the margin; they are boundary points of a classification. A vanishing determinant, a point at infinity, a null direction, a tangent contact, or an ideal boundary can all explain why a theorem needs the hypotheses it has. The code keeps those cases close enough to see, without pretending that a numerical experiment is a proof.


## Worked example philosophy

The worked examples favor small complete constructions over large opaque demonstrations. Every object is named, every coordinate convention is stated, and every saved artifact has a filename that names the mathematical idea it illustrates. A reader should be able to rerun a cell, change one number, and predict which part of the visual will move.

The examples also separate representation from interpretation. A conic matrix is not itself a conic until we decide which projective chart we are viewing. A sphere drawn on a flat page is not intrinsically flat. A hyperbolic disk uses Euclidean pixels to represent non-Euclidean distance. These separations are the main source of both power and confusion, so they are made explicit before the applied lab.


## How to read the artifacts

The artifacts in this notebook should be read as a small laboratory record. The PNG files are durable views of the construction, but the nearby code is part of the lesson: it states the coordinate convention, the parameter values, and the invariant being tested. The JSON and CSV files are intentionally plain so that a reader can open them outside the notebook and see the same evidence without rerunning every cell.

When a visual compares several objects, read it from the invariant outward. In this chapter the invariant is usually one of these: convex hull containment, signed area, monotone improvement of polygonal approximations, and Euler characteristic checks. Ask first what should stay fixed, then inspect which part of the figure changes. If the figure shows a family, the interesting information is often in the limiting member: a degenerate conic, a tangent contact, a boundary point, a null direction, or an approximation that is nearly smooth but still finite.

The saved checks do not replace proof. They play a different role. They protect the notebook from misleading pictures, record the numerical scale of residuals, and give the reader a concrete experiment to modify. A good modification changes the parameters while preserving the hypotheses; a better one also breaks a hypothesis and watches the check fail for a geometric reason.


## Applied lab

Approximate the unit disk with regular n-gons, watch the area error shrink, and compare that stable numerical behavior with the purely combinatorial invariant V - E + F.

For a stronger lab, change the parameters in two opposite directions: one that preserves the hypotheses and one that breaks them. The first change should keep the final checks small. The second should make a residual, signature, or visual feature fail in a meaningful way. That contrast is often where the real theorem becomes visible.


## Takeaways

- Convexity is best learned by moving between hulls, half-planes, and support functions.
- Area and volume formulas become trustworthy when paired with convergence experiments.
- Euler characteristic is combinatorial but survives geometric deformation of convex polyhedra.
- Rigidity and isoperimetry are global statements, so the notebook treats examples as probes rather than proofs by picture.


In [ ]:
assert generated_artifacts, "the notebook should generate artifacts"
assert_artifacts(generated_artifacts, min_bytes=32)
final_sanity = {
    "artifact_count": len(generated_artifacts),
    "all_artifacts_exist": all(path.exists() for path in generated_artifacts),
    "artifact_root": ARTIFACT_ROOT.relative_to(BOOK_ROOT).as_posix(),
}
final_sanity
